In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm
import matplotlib.pyplot as plt

# Cấu hình thiết bị (Ưu tiên GPU nếu có)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Tham số cấu hình
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 0.001
NUM_CLASSES = 7  # FER2013 có 7 loại cảm xúc

# Đường dẫn dataset trên Kaggle
DATA_DIR = '/kaggle/input/datasets/msambare/fer2013'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')

# Thư mục lưu Model sau khi train
OUTPUT_DIR = '/kaggle/working'
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, 'efficientnet_b2_fer2013.pth')

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.0, verbose=True):
        """
        Args:
            patience (int): Số lượng epoch tối đa chịu đựng khi chỉ số không cải thiện.
            min_delta (float): Ngưỡng thay đổi tối thiểu để được tính là có cải thiện.
            verbose (bool): In thông báo ra màn hình khi có sự thay đổi.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [ ]:
# Augmentation cho tập Train và Transform cho tập Test (Đã sửa lỗi ransforms -> transforms)
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((260, 260)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((260, 260)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Khởi tạo Dataset từ thư mục mã nguồn
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=data_transforms['train'])
test_dataset = datasets.ImageFolder(TEST_DIR, transform=data_transforms['test'])

# Khởi tạo DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples: {len(train_dataset)} | Classes: {train_dataset.classes}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Khởi tạo mô hình pre-trained EfficientNet-B2
model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)

# Thay đổi lớp Classifier cuối cùng để chống overfitting với Dropout=0.5
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.5, inplace=True),
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Theo dõi 'min' của Test Loss. Đã loại bỏ tham số verbose=True lỗi thời của PyTorch mới
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

# Khởi tạo bộ dừng sớm
early_stopping = EarlyStopping(patience=7, verbose=True)

In [ ]:
best_acc = 0.0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 10)
    
    # --- PHASE: TRAINING ---
    model.train()
    running_loss = 0.0
    running_corrects = 0
    
    train_bar = tqdm(train_loader, desc=f"Training Epoch {epoch+1}")
    for inputs, labels in train_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        
        # Cập nhật thông tin trên tqdm gbar
        train_bar.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)
    print(f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
    
    # --- PHASE: VALIDATION / TEST ---
    model.eval()
    val_loss = 0.0
    val_corrects = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(test_dataset)
    val_acc = val_corrects.double() / len(test_dataset)
    print(f"Test Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    
    # Cập nhật Learning Rate (Đã đặt đúng vị trí sau khi có val_loss)
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < old_lr:
        print(f"==> Tự động giảm Learning Rate từ {old_lr} xuống {new_lr}")
    
    # Lưu lại model tốt nhất dựa trên Test Accuracy tại thư mục /kaggle/working
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"=> Đã lưu checkpoint mô hình tốt nhất với Acc: {best_acc:.4f}")
        
    # Gọi kiểm tra Early Stopping bằng cách truyền Test Loss vào
    early_stopping(val_loss)
    
    # Nếu cờ dừng được bật -> Thoát vòng lặp
    if early_stopping.early_stop:
        print(f"\n[Early Stopping] Quá trình huấn luyện kết thúc sớm tại Epoch {epoch+1} để tránh Overfitting!")
        break

print(f"\nHuấn luyện hoàn tất! Accuracy tốt nhất trên tập Test: {best_acc:.4f}")

In [ ]:
if os.path.exists(MODEL_SAVE_PATH):
    print(f"Thành công! File trọng số đã được lưu tại: {MODEL_SAVE_PATH}")
    print(f"Kích thước file: {os.path.getsize(MODEL_SAVE_PATH) / (1024*1024):.2f} MB")
else:
    print("Chưa tìm thấy file model. Vui lòng kiểm tra lại quá trình train.")